# Exercise 2: Homoglyph detection

*Scripts and Encoding · version 0.1.0*

> Write a Python function `detect_homoglyphs(text)` that returns a list of (position, character, codepoint) tuples where the character is a Cyrillic/Latin homoglyph (use the table in the Unicode section). Test it on a string that mixes Cyrillic `а` (U+0430) and Latin `a` (U+0061).

**Setup.** Standard library only.

In [ ]:
import unicodedata

# Cyrillic letter -> Latin look-alike, from the chapter's homoglyph table
CYR_TO_LAT = {"а": "a", "е": "e", "о": "o", "р": "p", "ә": "ə", "і": "i", "һ": "h", "ј": "j"}
HOMOGLYPHS = set(CYR_TO_LAT) | set(CYR_TO_LAT.values())

def script_of(ch):
    name = unicodedata.name(ch, "")
    return "Cyrillic" if name.startswith("CYRILLIC") else "Latin" if name.startswith("LATIN") else "other"

def detect_homoglyphs(text):
    """(position, character, code point) for every character that has a look-alike in the other script."""
    return [(i, ch, f"U+{ord(ch):04X}") for i, ch in enumerate(text) if ch in HOMOGLYPHS]

def suspicious_words(text):
    """Words that mix Cyrillic and Latin letters: the case where a homoglyph is likely an error."""
    out = []
    for word in text.split():
        scripts = {script_of(c) for c in word if c.isalpha()}
        if {"Cyrillic", "Latin"} <= scripts:
            out.append((word, [(c, script_of(c)) for c in word if c in HOMOGLYPHS]))
    return out

text = "Алмaты мен Aстана, iPhone"        # Latin a in Алмaты, Latin A in Aстана
print(detect_homoglyphs(text))
print(suspicious_words(text))

## Worked solution

`detect_homoglyphs` reports every character that has a look-alike in the other script, including the legitimate Cyrillic *а* letters and the Latin letters of *iPhone*. On its own that is not a list of errors: most Cyrillic text consists largely of such letters.

The useful signal is a **word that mixes scripts**. `suspicious_words` finds *Алмaты* (Latin *a*, U+0061, inside a Cyrillic word) and *Aстана* (Latin capital *A*, U+0041; add the capitals to the table to catch them as homoglyphs too), while *iPhone*, an all-Latin word, is not flagged. Whether to repair a flagged word is a separate decision: the chapter's `suggest_cyrillic_token` proposes a repair only when the repaired form is a known word, and never edits the document.

## The recurring sentence

The same check on the book's recurring Kazakh sentence (Turkic World chapter), once clean and once with the copy-and-paste error used in the Scripts chapter's Hands-On.

In [ ]:
clean = "Қазақстан осы өңірдегі бейбітшілікті қолдайды."      # the recurring Kazakh sentence
corrupt = clean.replace("а", "a", 1)                            # one Latin a (U+0061) instead of Cyrillic а
print("clean:    ", suspicious_words(clean))
print("corrupted:", suspicious_words(corrupt))

Recorded output (20 October 2026):

```
clean:     []
corrupted: [('Қaзақстан', [('a', 'Latin'), ('а', 'Cyrillic'), ('а', 'Cyrillic')])]
```

The clean sentence has no mixed-script word; the corrupted one is caught at the word level, although whole-text script detection still calls it Cyrillic and the CTS view hides the error (see the chapter's Hands-On).